# 개인실습 — 조회 패턴 변경과 캐시 (7교시)

> `NoSQL_Redis_2_핵심개념_데모.ipynb`에서 만든 `products`·`view_history`(PostgreSQL), `users_embedded`(mongomock), `recent:u1001`(Redis List) 등을 그대로 이어서 씁니다. **이 노트북을 열기 전에 데모 노트북을 먼저 전부 실행해 두세요** — Step 1~4의 테이블·키가 이미 만들어져 있어야 아래 실습이 됩니다.
>
> 도전 1(태그 검색 3종) · 도전 2(캐시 레이어) · 도전 3(캐시 무효화), 총 7문제입니다. 빈칸은 각 문제 바로 아래 빈 코드 셀에 직접 작성하세요. 힌트 먼저, 그다음 정답은 별도 파일 `NoSQL_Redis_3_개인실습_정답.ipynb`에 있습니다.


In [7]:
# ✅ 환경 준비 — NoSQL_Redis_2_핵심개념_데모.ipynb의 헬퍼와 동일합니다 (파일이 분리되어 있어 다시 정의합니다)
import subprocess, json, time, io, os
import pandas as pd
import mongomock

# 💡 이 노트북을 단독으로 열어도 동작하도록, db-pg 시작 + db-redis 생성(없을 때만)·시작을 여기서 보장합니다.
if subprocess.run(["docker", "inspect", "db-redis"], capture_output=True, text=True).returncode != 0:
    subprocess.run(["docker", "run", "-d", "--name", "db-redis", "-p", "6379:6379", "redis:8"],
                    capture_output=True, text=True)
subprocess.run(["docker", "start", "db-pg", "db-redis"], capture_output=True, text=True)

def run_sql(sql, as_df=True):
    """psql subprocess 헬퍼 — db-pg 컨테이너 안의 psql로 실제 PostgreSQL(course_db)에 실행합니다."""
    if as_df:
        cmd = ["docker", "exec", "-e", "PGPASSWORD=postgres", "db-pg", "psql", "-U", "postgres", "-d", "course_db",
               "-v", "ON_ERROR_STOP=1", "-P", "footer=off", "-A", "-F", ",", "-c", sql]
        result = subprocess.run(cmd, capture_output=True, encoding="utf-8", errors="replace")
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        text = result.stdout.strip("\n")
        return pd.read_csv(io.StringIO(text)) if text else pd.DataFrame()
    else:
        cmd = ["docker", "exec", "-e", "PGPASSWORD=postgres", "db-pg", "psql", "-U", "postgres", "-d", "course_db",
               "-v", "ON_ERROR_STOP=1", "-c", sql]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        return result.stdout.strip()

def run_redis(*args):
    """redis-cli subprocess 헬퍼 — db-redis 컨테이너 안의 redis-cli로 실제 Redis에 실행합니다."""
    cmd = ["docker", "exec", "db-redis", "redis-cli"] + [str(a) for a in args]
    result = subprocess.run(cmd, capture_output=True, encoding="utf-8", errors="replace")
    out = result.stdout.strip()
    return None if out == "(nil)" else out

mongo_client = mongomock.MongoClient()
mongo_db = mongo_client["course_db"]

print("PostgreSQL 연결 확인:", not run_sql("SELECT 1;", as_df=False) == "")
print("Redis 연결 확인:", run_redis("ping") == "PONG")
print("mongomock 준비 완료:", mongo_db.name)

PostgreSQL 연결 확인: True
Redis 연결 확인: True
mongomock 준비 완료: course_db


## 도전 1 — 조회 패턴 변경: "태그로 문서 검색"

### 도전 1-A: PostgreSQL에서 태그 검색

🧪 **문제 1** — `products` 테이블에 `tags TEXT[]` 컬럼을 추가하고, 상품 501에 태그 `['전자기기','베스트셀러']`를 넣은 뒤, `'전자기기'` 태그가 있는 상품을 조회하세요.

💡 힌트: `ALTER TABLE ... ADD COLUMN IF NOT EXISTS tags TEXT[] DEFAULT '{}'` → `UPDATE ... SET tags = ARRAY[...]` → `WHERE '전자기기' = ANY(tags)`
👇 아래 빈 셀에 직접 작성해 보세요


In [ ]:
#SQL문 많이 안 써봤으면 어려운 문제
run_sql("""
ALTER TABLE products ADD COLUMN IF NOT EXISTS tags TEXT[] DEFAULT '{}';
UPDATE products SET tags = ARRAY['전자기기', '베스트셀러'] WHERE id = 501;
""", as_df=False)  # as_df=False: 결과를 표(DataFrame)로 받을 필요 없는 명령(CREATE/INSERT)이므로 텍스트로만 받음
print(run_sql("""SELECT id, name, tags FROM products WHERE '전자기기' = ANY(tags);"""))  # 여긴 as_df 기본값(True) 그대로 사용 -> 결과가 pandas 표로 예쁘게 출력됨

         id   name    tags
501  무선 이어폰  {전자기기  베스트셀러}


🧪 **문제 2** — 방금 만든 `tags` 컬럼에 GIN 인덱스를 추가하고, `EXPLAIN`으로 실제 실행 계획을 확인하세요.

💡 힌트: `CREATE INDEX ... USING GIN(tags)` → `EXPLAIN SELECT ...`. 3행짜리 작은 테이블이라 인덱스가 있어도 플래너가 Seq Scan을 고를 수 있습니다 — 9/22 pgvector에서 본 것과 같은 교훈입니다.
👇 아래 빈 셀에 직접 작성해 보세요


In [10]:
#SQL문 많이 안 써봤으면 어려운 문제
run_sql("CREATE INDEX IF NOT EXISTS idx_products_tags ON products USING GIN(tags);", as_df=False)
print(run_sql("EXPLAIN SELECT id, name FROM products WHERE '전자기기' = ANY(tags);"))

                                          QUERY PLAN
0  Seq Scan on products  (cost=0.00..1.07 rows=1 ...
1                Filter: ('전자기기'::text = ANY (tags))


### 도전 1-B: MongoDB(mongomock)에서 태그 검색

🧪 **문제 3** — `products_doc` 컬렉션에 `{"_id": "P501", "name": "무선 이어폰", "tags": ["전자기기","베스트셀러"]}`를 넣고, 태그로 검색하는 함수를 만들어 `"전자기기"`로 조회해 보세요.

💡 힌트: `insert_one({...})` → `find({"tags": tag})`는 배열에 값이 "포함"되어 있으면 자동으로 매치합니다(완전 일치가 아님) — pymongo의 실제 동작입니다.
👇 아래 빈 셀에 직접 작성해 보세요


In [ ]:
products_doc = mongo_db["products_doc"]
#복습할 때는 insert_many를 활용해서 여러 개의 값을 넣고 결과를 확인해보자
products_doc.insert_one({"_id": "P501", "name": "무선 이어폰", "tags": ["전자기기", "베스트셀러"]})

def mongo_docs_by_tag(tag: str) -> list:
    """mongomock: 태그 배열에 특정 값이 있는 문서 조회"""
    # find({"tags": tag})처럼 배열 필드에 단일 값을 조건으로 주면, MongoDB는 자동으로
    # "그 배열 안에 이 값이 포함된 문서"를 찾아줍니다(PostgreSQL의 ANY(tags)와 같은 효과를 다른 문법으로 표현).
    return list(products_doc.find({"tags": tag}))

print("mongomock 태그 검색 ->", mongo_docs_by_tag("전자기기"))

mongomock 태그 검색 -> [{'_id': 'P501', 'name': '무선 이어폰', 'tags': ['전자기기', '베스트셀러']}]


### ⚠️ 잠깐 — 직접 매처를 짰다면 어떻게 될까? (읽고 이해하기, 작성 불필요)

아래 셀은 직접 작성할 필요 없이 **실행하고 결과만 확인**하세요. 만약 mongomock/실제 MongoDB 대신 필드값을 "완전히 같은지"만 비교하는 순수 파이썬 매처를 직접 짰다면 어떻게 될지 보여줍니다.


In [16]:
# ⚠️ 흔한 실수(개념 확인) — 직접 매처를 짰다면 어떻게 될까?
# 이전 버전의 FakeMongoCollection처럼 필드값을 "완전히 같은지"만 비교하는 순수 파이썬 매처를 짰다면:
def naive_match(doc, query):
    for k, v in query.items():
        if doc.get(k) != v:      # 배열 "포함"이 아니라 "완전 일치"만 검사
            return False
    return True

sample = {"_id": "P501", "name": "무선 이어폰", "tags": ["전자기기", "베스트셀러"]}
naive_result = naive_match(sample, {"tags": "전자기기"})
print("순수 동등비교 매처였다면 매치 성공?", naive_result)
print("-> 배열 전체(['전자기기','베스트셀러'])와 문자열('전자기기')을 통째로 비교하므로 항상 False입니다.")
print("-> mongomock/실제 MongoDB는 이 배열-포함 검사를 드라이버가 대신 해주기 때문에 정확히 동작합니다.")
print("-> 이래서 실무에서도, 이 실습에서도 검증된 라이브러리를 쓰지 직접 매칭 로직을 짜지 않습니다.")

순수 동등비교 매처였다면 매치 성공? False
-> 배열 전체(['전자기기','베스트셀러'])와 문자열('전자기기')을 통째로 비교하므로 항상 False입니다.
-> mongomock/실제 MongoDB는 이 배열-포함 검사를 드라이버가 대신 해주기 때문에 정확히 동작합니다.
-> 이래서 실무에서도, 이 실습에서도 검증된 라이브러리를 쓰지 직접 매칭 로직을 짜지 않습니다.


### 도전 1-C: Redis에서 태그 검색

🧪 **문제 4** — Redis Set으로 태그별 상품 집합을 관리하세요: `tag:전자기기:products`에 `product:501`·`product:502`를, `tag:베스트셀러:products`에 `product:501`을 넣고, `"전자기기"` 태그의 상품 목록을 조회하는 함수를 만드세요.

💡 힌트: `SADD`(추가) → `SMEMBERS`(전체 조회). `run_redis()`가 돌려주는 텍스트는 줄바꿈 문자열이므로 `.splitlines()`가 필요합니다.
👇 아래 빈 셀에 직접 작성해 보세요


In [13]:
def redis_setup_tag_index():
    run_redis("sadd", "tag:전자기기:products", "product:501", "product:502")
    run_redis("sadd", "tag:베스트셀러:products", "product:501")  # product:501은 두 태그 Set에 모두 들어감(중복 가능)
def redis_docs_by_tag(tag: str) -> list:
    key = f"tag:{tag}:products"
    result = run_redis("smembers", key)
    return result.splitlines() if result else []

run_redis("del", "tag:전자기기:products", "tag:베스트셀러:products")
redis_setup_tag_index()
print("전자기기 태그 상품 ->", redis_docs_by_tag("전자기기"))

전자기기 태그 상품 -> ['product:501', 'product:502']


## 도전 2 · Redis 캐시 레이어 추가

🧪 **문제 5** — `view_history`에서 사용자의 최근 조회를 가져오되, Redis를 캐시로 앞에 두세요: 캐시에 있으면 캐시에서, 없으면 PostgreSQL에서 가져와 캐시에 저장(TTL 60초)하는 함수를 만들고, 연속 3번 호출했을 때 첫 번째만 `db`, 나머지는 `cache`에서 오는지 확인하세요.

💡 힌트: 캐시 키는 `f"recent_views:{user_id}:{limit}"` 형태로 만드세요. `run_redis("get", key)` → 있으면 `json.loads`로 반환, 없으면 PostgreSQL 조회 후 `run_redis("set", key, ..., "EX", "60")`.
👇 아래 빈 셀에 직접 작성해 보세요


In [14]:
def pg_recent_views_raw(user_id: int, limit: int = 5):
    df = run_sql(f"""
        SELECT user_id, product_id, viewed_at
        FROM view_history WHERE user_id = {user_id}
        ORDER BY viewed_at DESC LIMIT {limit};
    """)
    return df.to_dict(orient="records")

def get_recent_views_cached(user_id: int, limit: int = 5, ttl: int = 60) -> dict:
    cache_key = f"recent_views:{user_id}:{limit}"
    cached = run_redis("get", cache_key)
    if cached:
        return {"source": "cache", "data": json.loads(cached)}
    data = pg_recent_views_raw(user_id, limit)  # 캐시에 없을 때만 위의 "느린 버전" 함수를 호출 (그래서 이 문제를 푸는 데 실질적으로 필요한 건 아래 함수 2개)
    run_redis("set", cache_key, json.dumps(data, ensure_ascii=False, default=str), "EX", str(ttl))
    return {"source": "db", "data": data}

def measure_cache_effect(user_id: int = 1001):
    cache_key = f"recent_views:{user_id}:5"
    run_redis("del", cache_key)  # 캐시를 비운 상태에서 시작 -> 첫 조회는 반드시 db를 타게 됨
    for i in range(3):  # 같은 요청을 3번 반복해서, 캐시가 쌓이면서 속도가 어떻게 달라지는지 관찰
        t0 = time.perf_counter()
        result = get_recent_views_cached(user_id)
        elapsed = (time.perf_counter() - t0) * 1000  # 초 -> 밀리초 단위 변환
        # f"{elapsed:6.1f}"는 전체 폭 6칸을 확보하고 소수점 첫째 자리까지 표시(숫자가 짧으면 앞에 공백으로 채움)
        print(f"조회 {i+1}: {elapsed:6.1f}ms  [source: {result['source']}]")

measure_cache_effect()

조회 1:  315.1ms  [source: db]
조회 2:   91.3ms  [source: cache]
조회 3:   92.8ms  [source: cache]


🧪 **문제 6** — 캐시 없이 10번 조회한 시간과, 캐시를 쓰며 10번 조회한 시간을 각각 재서 비교하세요.

💡 힌트: `time.perf_counter()`로 전후 시각을 재고 밀리초로 환산하세요. 절대값보다 "캐시가 있을 때 훨씬 빠르다"는 방향과 비율이 중요합니다 — `run_sql`/`run_redis`가 매번 프로세스를 새로 띄우기 때문에 절대값 자체는 실무보다 크게 나옵니다.
👇 아래 빈 셀에 직접 작성해 보세요


In [15]:
# benchmark: "캐시 없이 매번 DB만 사용" vs "캐시를 사용" 두 방식의 속도 차이를 직접 재서 비교합니다.
def benchmark(n: int = 10, user_id: int = 1001):
    cache_key = f"recent_views:{user_id}:5"

    # 1) 캐시 없음 시나리오: 매 반복마다 캐시를 강제로 지운 뒤(del) 느린 함수(pg_recent_views_raw)만 호출
    t0 = time.perf_counter()
    for _ in range(n):
        run_redis("del", cache_key)
        pg_recent_views_raw(user_id)
    t_no_cache = (time.perf_counter() - t0) * 1000

    # 2) 캐시 있음 시나리오: 딱 한 번만 비우고, 나머지는 캐시가 쌓인 상태로 반복 호출
    run_redis("del", cache_key)
    t0 = time.perf_counter()
    for _ in range(n):
        get_recent_views_cached(user_id)
    t_with_cache = (time.perf_counter() - t0) * 1000

    print(f"캐시 없음 {n}회: {t_no_cache:.1f}ms")
    print(f"캐시 있음 {n}회: {t_with_cache:.1f}ms")
    # (1 - t_with_cache/t_no_cache)*100은 "캐시가 전체 시간을 몇 % 줄여줬는지" 계산하는 공식입니다
    # — 예를 들어 캐시 있음이 캐시 없음의 20%밖에 안 걸렸다면, 80% 절감된 것입니다.
    print(f"절감: {t_no_cache - t_with_cache:.1f}ms ({(1 - t_with_cache/t_no_cache)*100:.0f}%)")

benchmark()

캐시 없음 10회: 2077.4ms
캐시 있음 10회: 1087.7ms
절감: 989.7ms (48%)


## 도전 3 · 캐시 무효화

🧪 **문제 7** — 새로운 조회 기록을 PostgreSQL에 남기는 동시에, 해당 사용자의 캐시를 지우는(무효화) 함수를 만들고, 무효화 전/후로 캐시 소스가 바뀌는지 확인하세요.

💡 힌트: `INSERT INTO view_history ...` 실행 직후 `run_redis("del", f"recent_views:{user_id}:5")`. 무효화 후 다시 조회하면 캐시가 없으므로 `db`에서 와야 정상입니다.
👇 아래 빈 셀에 직접 작성해 보세요


In [ ]:
# record_view: 새 조회 기록을 남기면서, 이제는 낡은 정보가 된 캐시를 함께 지우는(=무효화하는) 함수
def record_view(user_id: int, product_id: str):
    """새 조회를 PostgreSQL에 기록하고 캐시를 무효화"""
    run_sql(f"""
        INSERT INTO view_history (user_id, product_id) VALUES ({user_id}, '{product_id}')
    """, as_df=False)
    # 캐시 무효화(invalidation): 데이터가 바뀌었으니 기존 캐시 값은 더 이상 맞지 않습니다.
    # 여기서는 새 값을 캐시에 직접 채워 넣지 않고, 그냥 지워버립니다(del) — 그러면 다음 조회 때
    # get_recent_views_cached()가 캐시가 없다고 판단해 자동으로 DB에서 최신 값을 다시 읽어와 캐시를 새로 채웁니다.
    run_redis("del", f"recent_views:{user_id}:5")

before = get_recent_views_cached(1001)
print("기록 전 캐시 소스:", before["source"], "| 개수:", len(before["data"]))

record_view(1001, "product:504")  # 새 조회 기록 -> 내부적으로 캐시가 삭제됨
after = get_recent_views_cached(1001)
# 캐시가 무효화됐으므로 이 호출은 반드시 "db"를 거쳐야 하고(캐시가 비어있으니), 그 결과 캐시가 다시 채워집니다.
print("새 조회 기록 후 캐시 소스:", after["source"], "(무효화됐으므로 db여야 함)")
# [d["product_id"] for d in after["data"]]는 딕셔너리 리스트에서 product_id 값만 뽑아 새 리스트로 만드는
# 리스트 컴프리헨션입니다 — 방금 기록한 product:504가 최신 목록 맨 앞(가장 최근)에 보여야 정상입니다.
print("최신 목록:", [d["product_id"] for d in after["data"]])

## 정리 및 자가체크

- [ ] 문제 1~2: PostgreSQL `TEXT[]`+GIN으로 태그 검색을 직접 만들었다
- [ ] 문제 3: mongomock 배열 필드 검색을 직접 만들었다 (+ 흔한 실수 이해)
- [ ] 문제 4: Redis Set으로 태그 인덱스를 직접 만들었다
- [ ] 문제 5~6: Cache-Aside 캐시 레이어를 직접 구현하고 속도 차이를 측정했다
- [ ] 문제 7: 캐시 무효화를 직접 구현했다

막혔던 부분은 `NoSQL_Redis_3_개인실습_정답.ipynb`에서 확인하세요.
